# Assignment 1 — Gradio dashboard

Loads the models and data saved by `train.ipynb` (nothing is retrained here)
and serves an interactive dashboard. Run with `uv run python main.py app`.

Plots are drawn with matplotlib on the server and sent to the browser as
images, so they don't depend on any JavaScript plotting library loading.

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import matplotlib
import numpy as np
import pandas as pd
import torch
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.figure import Figure

from features import SEASONS, build_features

matplotlib.use("Agg")

ARTIFACTS = Path("artifacts")
if not (ARTIFACTS / "config.json").exists():
    raise SystemExit("No trained artifacts found. Run `uv run python main.py train` first.")

MODEL_COLORS = {"scikit-learn": "#2a78d6", "manual PyTorch": "#eb6834", "standard PyTorch": "#1baf7a"}
MODEL_STYLES = {"scikit-learn": "-", "manual PyTorch": "--", "standard PyTorch": ":"}
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
BLUE, ACTUAL_COLOR = "#2a78d6", "#52514e"
INK, MUTED, GRID = "#0b0b0b", "#898781", "#e6e5e1"
DIVERGING = LinearSegmentedColormap.from_list("div", ["#eb6834", "#f0efec", "#2a78d6"])
matplotlib.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "font.size": 11,
})

## Load trained artifacts

In [ ]:
config = json.loads((ARTIFACTS / "config.json").read_text())
FEATURES, TARGET = config["features"], config["target"]

scaler = joblib.load(ARTIFACTS / "scaler.joblib")
sk_model = joblib.load(ARTIFACTS / "sklearn_ridge.joblib")
manual_params = torch.load(ARTIFACTS / "manual_pytorch.pt")


class LinearRegressionModel(torch.nn.Module):
    """Same architecture as in train.ipynb, so the saved state_dict loads into it."""

    def __init__(self, n_features):
        super().__init__()
        self.linear = torch.nn.Linear(n_features, 1)

    def forward(self, x):
        return self.linear(x)


std_model = LinearRegressionModel(len(FEATURES))
std_model.load_state_dict(torch.load(ARTIFACTS / "standard_pytorch.pt"))
std_model.eval()

df = pd.read_csv(ARTIFACTS / "data_with_split.csv", parse_dates=["date"])
test_metrics = pd.read_csv(ARTIFACTS / "test_metrics.csv")
coefficients = pd.read_csv(ARTIFACTS / "coefficients.csv", index_col=0)
loss_curves = pd.read_csv(ARTIFACTS / "loss_curves.csv", index_col="epoch")

## Prediction with the loaded models

In [ ]:
def predict(rows: pd.DataFrame) -> dict[str, np.ndarray]:
    """Predict bikes per hour for cleaned rows with each of the three models."""
    X = scaler.transform(build_features(rows)[FEATURES])
    Xt = torch.tensor(X, dtype=torch.float32)
    with torch.no_grad():
        manual = (Xt @ manual_params["w"] + manual_params["b"]).squeeze(1).numpy()
        standard = std_model(Xt).squeeze(1).numpy()
    raw = {"scikit-learn": sk_model.predict(X), "manual PyTorch": manual, "standard PyTorch": standard}
    return {name: np.clip(np.expm1(p) if TARGET == "log" else p, 0, None) for name, p in raw.items()}


test_df = df[df["split"] == "test"].reset_index(drop=True)
test_preds = predict(test_df)
TEST_DATES = sorted(test_df["date"].dt.strftime("%Y-%m-%d").unique())

## Plots

In [ ]:
def new_figure(width=11, height=4.2, **subplot_kw):
    """A pyplot-free Figure, so repeated callbacks never accumulate open figures."""
    fig = Figure(figsize=(width, height), dpi=100, layout="constrained")
    return fig, fig.subplots(**subplot_kw)


def pred_vs_actual_plot():
    fig, axes = new_figure(13, 4.6, ncols=3, sharey=True)
    top = max(test_df["count"].max(), max(p.max() for p in test_preds.values()))
    for ax, (name, pred) in zip(axes, test_preds.items()):
        ax.scatter(test_df["count"], pred, s=8, alpha=0.35, color=MODEL_COLORS[name], linewidths=0)
        ax.plot([0, top], [0, top], color=MUTED, linestyle="--", linewidth=1.2, label="perfect prediction")
        ax.set(title=name, xlabel="actual bikes / hour", xlim=(0, top), ylim=(0, top))
    axes[0].set_ylabel("predicted bikes / hour")
    axes[0].legend(frameon=False, loc="upper left")
    fig.suptitle("Predicted vs. actual — test set")
    return fig


def residual_plot():
    fig, ax = new_figure()
    bins = np.linspace(-2000, 2000, 81)
    for name, pred in test_preds.items():
        ax.hist(pred - test_df["count"], bins=bins, histtype="step", linewidth=2,
                linestyle=MODEL_STYLES[name], color=MODEL_COLORS[name], label=name)
    ax.axvline(0, color=MUTED, linewidth=1)
    ax.set(title="Prediction error on test set (predicted − actual)",
           xlabel="bikes / hour (negative = under-prediction)", ylabel="hours")
    ax.legend(frameon=False)
    return fig


def day_profile_plot(date):
    day = test_df[test_df["date"].dt.strftime("%Y-%m-%d") == date]
    fig, ax = new_figure()
    ax.plot(day["hour"], day["count"], color=ACTUAL_COLOR, linewidth=3, marker="o", markersize=5, label="actual")
    for name, pred in test_preds.items():
        ax.plot(day["hour"], pred[day.index], color=MODEL_COLORS[name], linewidth=2,
                linestyle=MODEL_STYLES[name], label=name)
    row = day.iloc[0]
    ax.set(title=f"{date} ({row['season']}, {'weekend' if row['weekend'] else 'weekday'}, {row['holiday']})",
           xlabel="hour of day", ylabel="bikes rented", xticks=range(24))
    ax.set_ylim(bottom=0)
    ax.legend(frameon=False, ncols=4, loc="upper left")
    return fig


def loss_plot():
    fig, ax = new_figure(height=3.8)
    for name in loss_curves.columns:
        ax.plot(loss_curves.index, loss_curves[name], color=MODEL_COLORS[name],
                linewidth=2, linestyle=MODEL_STYLES[name], label=name)
    ax.set(title="PyTorch training loss", xlabel="epoch", ylabel="loss (log scale)", yscale="log")
    ax.legend(frameon=False)
    return fig


DIST_COLUMNS = ["count", "hour", "temp", "solar", "rain", "snow",
                "humidity", "wind", "visibility", "dew_point"]


def distribution_plot(column, by_split):
    fig, ax = new_figure()
    if by_split:
        bins = np.histogram_bin_edges(df[column], bins=40)
        for split, color in SPLIT_COLORS.items():
            ax.hist(df.loc[df["split"] == split, column], bins=bins, density=True,
                    histtype="step", linewidth=2, color=color, label=split)
        ax.set_ylabel("density")
        ax.legend(frameon=False)
    else:
        ax.hist(df[column], bins=40, color=BLUE, edgecolor="#fcfcfb", linewidth=1)
        ax.set_ylabel("hours")
    ax.set(title=f"Distribution of {column}", xlabel=column)
    return fig


def target_vs_feature_plot(column):
    fig, ax = new_figure()
    ax.scatter(df[column], df["count"], s=6, alpha=0.25, color=BLUE, linewidths=0)
    ax.set(title=f"Rented bikes vs. {column}", xlabel=column, ylabel="bikes / hour")
    return fig


def correlation_plot():
    cols = ["count", "hour", "temp", "humidity", "wind", "visibility",
            "dew_point", "solar", "rain", "snow", "weekend"]
    corr = df[cols].corr()
    fig, ax = new_figure(9, 7.5)
    im = ax.imshow(corr, cmap=DIVERGING, vmin=-1, vmax=1)
    ax.set_xticks(range(len(cols)), cols, rotation=45, ha="right")
    ax.set_yticks(range(len(cols)), cols)
    ax.grid(False)
    for i in range(len(cols)):
        for j in range(len(cols)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=8, color=INK)
    fig.colorbar(im, ax=ax, shrink=0.8, label="Pearson correlation")
    ax.set_title("Correlation matrix (all original numeric columns)")
    return fig


def what_if(hour, weekend, holiday, season, temp, solar, rain, snow):
    row = pd.DataFrame([{
        "hour": hour, "weekend": int(weekend), "holiday": "Holiday" if holiday else "No Holiday",
        "season": season, "temp": temp, "solar": solar, "rain": rain, "snow": snow,
    }])
    preds = predict(row)
    return pd.DataFrame({"model": list(preds), "predicted bikes / hour": [round(float(p[0])) for p in preds.values()]})

## Dashboard layout

In [ ]:
with gr.Blocks(title="Bike demand models") as demo:
    gr.Markdown(
        "# Hourly bike-rental demand\n"
        f"Linear regression (target: `{TARGET}` count, Ridge α = {config['alpha']}) trained three ways. "
        "All plots use the saved models and data from `main.py train`; nothing is retrained.")

    with gr.Tab("Model comparison"):
        gr.Markdown("### Test-set metrics (bikes per hour)")
        gr.Dataframe(test_metrics, interactive=False)
        gr.Plot(pred_vs_actual_plot())
        gr.Plot(residual_plot())

    with gr.Tab("Single test day"):
        date_input = gr.Dropdown(TEST_DATES, value=TEST_DATES[0], label="Test-set day")
        day_plot = gr.Plot(day_profile_plot(TEST_DATES[0]))
        date_input.change(day_profile_plot, date_input, day_plot)

    with gr.Tab("Distributions"):
        with gr.Row():
            column_input = gr.Dropdown(DIST_COLUMNS, value="temp", label="Variable")
            split_input = gr.Checkbox(label="Compare train / val / test", value=False)
        dist_plot = gr.Plot(distribution_plot("temp", False))
        scatter_plot = gr.Plot(target_vs_feature_plot("temp"))
        column_input.change(distribution_plot, [column_input, split_input], dist_plot)
        split_input.change(distribution_plot, [column_input, split_input], dist_plot)
        column_input.change(target_vs_feature_plot, column_input, scatter_plot)
        gr.Plot(correlation_plot())

    with gr.Tab("Agreement & training"):
        gr.Markdown("### Coefficients (on standardised features)")
        gr.Dataframe(coefficients.round(4).reset_index(names="feature"), interactive=False)
        gr.Markdown("### Agreement with scikit-learn\n" + "\n".join(
            f"- {k}: `{v:.6f}`" for k, v in config["agreement"].items()))
        gr.Plot(loss_plot())

    with gr.Tab("What-if prediction"):
        with gr.Row():
            with gr.Column():
                hour_in = gr.Slider(0, 23, value=8, step=1, label="Hour of day")
                weekend_in = gr.Checkbox(label="Weekend", value=False)
                holiday_in = gr.Checkbox(label="Holiday", value=False)
                season_in = gr.Dropdown(SEASONS, value="Spring", label="Season")
            with gr.Column():
                temp_in = gr.Slider(-20, 40, value=15, step=0.5, label="Temperature (°C)")
                solar_in = gr.Slider(0, 3.6, value=0.5, step=0.05, label="Solar radiation (MJ/m²)")
                rain_in = gr.Slider(0, 35, value=0, step=0.5, label="Rainfall (mm)")
                snow_in = gr.Slider(0, 9, value=0, step=0.1, label="Snowfall (cm)")
        inputs = [hour_in, weekend_in, holiday_in, season_in, temp_in, solar_in, rain_in, snow_in]
        what_if_out = gr.Dataframe(what_if(8, False, False, "Spring", 15, 0.5, 0, 0), interactive=False)
        for component in inputs:
            component.change(what_if, inputs, what_if_out)

In [ ]:
demo.launch(inbrowser=True)